# Socratic Debug Tutor — QLoRA fine-tune on a free Colab T4

This notebook runs **one** experiment end to end:

> Does QLoRA fine-tuning on the full 600-example Dataset V1 make Qwen3-1.7B hold
> the Socratic Debug Tutor behavior more reliably than the untouched base model?

It trains **N=600 only**, then evaluates **base vs tuned** on the same held-out
scenarios, with the same prompt, the same generation settings and the same
judge. The only difference between the two models is the adapter.

### What this notebook deliberately does NOT do

* It does **not** run the N=125 / 250 / 500 data-efficiency sweep. That stays
  gated on this result being worth extending.
* It does **not** generate or modify data. Dataset V1 is frozen, hashed, and
  committed in the repository — the notebook verifies it rather than rebuilding it.
* It does **not** tune hyperparameters. This is a data → behavior experiment;
  changing the recipe after seeing a score would confound it.

### Before you start

1. **Runtime → Change runtime type → T4 GPU.** Training will refuse to start
   on CPU or on a pre-Turing GPU.
2. **Push the branch** that contains the frozen recipe (see the clone cell).
3. Add `ANTHROPIC_API_KEY` in the 🔑 **Secrets** panel on the left. Needed only
   by the evaluation cells at the end — training itself needs no credentials.

### Roughly what to expect

| Phase | Time on a T4 |
| --- | --- |
| Install + verify | ~3 min |
| Training, 540 examples × 3 epochs | ~35–60 min |
| Base + tuned evaluation, 40 generations + judging | ~15–25 min |

Colab disconnects idle sessions. Keep the tab open, and run the *Save to Drive*
cell before the evaluation step so a dropped session does not cost you the
adapter.

## 1. Confirm the GPU can actually do 4-bit

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

import torch

assert torch.cuda.is_available(), (
    "No CUDA device. Runtime -> Change runtime type -> T4 GPU, then rerun."
)

props = torch.cuda.get_device_properties(0)
cc = (props.major, props.minor)
vram = props.total_memory / 2**30

print(f"\ntorch {torch.__version__}")
print(f"{props.name} | compute capability {cc[0]}.{cc[1]} | {vram:.1f} GiB")

# bitsandbytes NF4 kernels need Turing or newer. T4 = 7.5, L4 = 8.9, A100 = 8.0.
assert cc >= (7, 5), (
    f"Compute capability {cc[0]}.{cc[1]} is too old for 4-bit NF4 (need >= 7.5). "
    f"Switch the runtime to a T4."
)
assert vram >= 12, f"Only {vram:.1f} GiB of VRAM; this run needs ~12 GiB."

# A T4 has no bfloat16. The T4 config accounts for that; this just reports it.
print(f"bfloat16 supported: {torch.cuda.is_bf16_supported()}")
print("\nGPU OK.")

## 2. Get the repository

The frozen recipe lives on a branch. **Push it first** from your machine:

```bash
git push -u origin n600-training-prep
```

Set `BRANCH` below to `master` once that branch is merged.

In [ ]:
REPO   = "https://github.com/sohailataiml/SMLqLORA.git"
BRANCH = "n600-training-prep"   # <- change to "master" after merging

import os, subprocess, sys
from pathlib import Path

WORKDIR = Path("/content/SMLqLORA")

if not WORKDIR.exists():
    !git clone --branch {BRANCH} --depth 1 {REPO} {WORKDIR}
else:
    print(f"{WORKDIR} already present - reusing it")

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

commit = subprocess.run(["git", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"\ncwd    : {os.getcwd()}")
print(f"branch : {BRANCH}")
print(f"commit : {commit}")

assert Path("data/versions/v1/selected.jsonl").exists(), (
    "Dataset V1 is missing. You are on a branch that predates the freeze - "
    "check BRANCH above."
)
assert Path("training/configs/qlora_qwen3_1_7b_t4.yaml").exists(), (
    "T4 config missing. Push the branch that contains it."
)
print("\nRepository OK.")

## 3. Install the training stack

Colab already ships a CUDA build of torch, so it is deliberately **not**
reinstalled — that download is multi-GB and often lands on a wheel mismatched to
the runtime's driver.

The version floors matter. `training/train.py` calls `SFTConfig(max_length=...)`
and `SFTTrainer(processing_class=...)`, which are TRL ≥ 0.20 spellings; older TRL
fails with an opaque `TypeError` several minutes in. The cell after the install
checks the API surface directly rather than trusting a version string.

In [ ]:
# Colab preinstalls torchao 0.10.0. PEFT's LoRA dispatcher calls
# is_torchao_available(), which RAISES on anything below 0.16.0 rather than
# returning False - so PeftModel.from_pretrained fails and the tuned model
# returns an empty string for every scenario. Nothing here uses torchao.
!pip uninstall -y -q torchao

!pip install -q -r requirements-colab.txt

In [ ]:
# Restart-free capability check: assert the API this repo actually calls exists.
import inspect

import transformers, peft, trl, bitsandbytes, accelerate, datasets
from trl import SFTConfig, SFTTrainer

print(f"transformers  {transformers.__version__}")
print(f"trl           {trl.__version__}")
print(f"peft          {peft.__version__}")
print(f"bitsandbytes  {bitsandbytes.__version__}")
print(f"accelerate    {accelerate.__version__}")
print(f"datasets      {datasets.__version__}")

sft_params = inspect.signature(SFTConfig.__init__).parameters
trainer_params = inspect.signature(SFTTrainer.__init__).parameters

assert "max_length" in sft_params, (
    "This TRL is too old: SFTConfig has no `max_length` (it was `max_seq_length` "
    "before 0.20). Run:  pip install -q -U 'trl>=0.20'  and rerun this cell."
)
assert "processing_class" in trainer_params, (
    "This TRL is too old: SFTTrainer has no `processing_class`. "
    "Run:  pip install -q -U 'trl>=0.20'  and rerun this cell."
)

print("\nTRL API surface OK.")

# PEFT injects LoRA by walking a list of dispatchers, and some of them raise
# on an incompatible optional dependency instead of declining. A stale
# torchao in the image is enough to make every adapter load fail, which shows
# up as a tuned model that answers nothing. Prove injection works now.
try:
    from peft.import_utils import is_torchao_available
    is_torchao_available()
except ImportError as exc:
    raise AssertionError(
        'PEFT cannot load adapters in this environment: ' + str(exc)
        + '   Fix:  !pip uninstall -y torchao'
    ) from exc
except Exception:
    pass  # any other outcome means the dispatcher declines, which is fine

print('PEFT adapter injection OK.')

## 4. Verify the frozen dataset before spending GPU time

This is the offline gate. It refuses to pass unless the data on disk still
hashes to the frozen Dataset V1 value, converts to exactly 600 well-formed chat
records, carries the **weak** system prompt, leaks no quality-gate metadata into
model-visible text, and overlaps no evaluation split.

If this fails, stop. Do not "fix" it by editing Dataset V1 — V1 is immutable, and
any genuine correction is Dataset V2.

In [ ]:
!python scripts/verify_training_data.py \
    --config training/configs/qlora_qwen3_1_7b_t4.yaml \
    --expect-count 600

## 5. Check that training and inference agree on the chat template

Qwen3's template emits a `<think>…</think>` block, and whether it appears depends
on *how* the template is called. Training renders a full conversation; evaluation
renders a prompt with `add_generation_prompt=True, enable_thinking=False`. If
those two disagree, the tuned model is scored on a prefix it never saw in
training — and the damage would look like a data problem.

They should agree: the template's `loop.last` branch emits the same empty think
block that inference prefills. This asserts it, so a future tokenizer revision
cannot break it silently.

In [ ]:
from transformers import AutoTokenizer

from training.dataset import build_dataset, load_accepted
from training.train import TrainingConfig

cfg = TrainingConfig.load("training/configs/qlora_qwen3_1_7b_t4.yaml")
model_cfg = cfg.section("model")

tok = AutoTokenizer.from_pretrained(
    model_cfg["base_model"], revision=str(model_cfg["revision"])
)

examples = load_accepted(cfg.section("data")["accepted_path"])
split = build_dataset(examples, validation_fraction=0.1)
record = split.train[0]

# How TRL will render this example during training.
train_text = tok.apply_chat_template(record["messages"], tokenize=False)

# How the evaluator will render the same conversation at inference.
eval_prompt = tok.apply_chat_template(
    record["messages"][:-1],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

assert train_text.startswith(eval_prompt), (
    "TRAIN/INFERENCE TEMPLATE MISMATCH.\n"
    "The training text does not begin with the exact prompt the evaluator "
    "builds, so the model would be trained and scored on different prefixes.\n\n"
    f"--- eval prompt tail ---\n{eval_prompt[-300:]}\n\n"
    f"--- training text at that point ---\n{train_text[:len(eval_prompt)][-300:]}"
)

target = train_text[len(eval_prompt):]
print("Prefixes match. The model is trained on exactly what it is asked at eval.\n")
print("--- prompt tail ---")
print(eval_prompt[-220:])
print("\n--- supervised target ---")
print(target[:400])

## 6. Dry run

Builds the dataset, re-checks contamination and writes checkpoint metadata
without loading a model. Cheap; catches configuration mistakes before the GPU
time starts.

In [ ]:
!python -m training.train \
    --config training/configs/qlora_qwen3_1_7b_t4.yaml \
    --dry-run --run-name socratic-v1-n600-dry

## 7. Train — N=600

540 training examples, 60 validation, 3 epochs, effective batch size 16.
Roughly **35–60 minutes** on a T4.

The log is tee'd to a file so an interrupted session still leaves evidence, and
VRAM is sampled in the background because peak usage is not observable from this
process once training runs in a subprocess.

**Do not change the recipe if the loss looks unusual.** Diagnose first; the
answer to a bad result is Dataset V2, not different hyperparameters.

In [ ]:
import subprocess, time
from pathlib import Path

RUN = "socratic-v1-n600"
Path("results/training").mkdir(parents=True, exist_ok=True)
LOG = f"results/training/{RUN}.log"
VRAM = "/content/vram_samples.log"

!rm -f {VRAM}
# Sample GPU memory every 5s; peak is not observable from this process once
# training runs in a subprocess.
get_ipython().system_raw(
    f"nohup bash -c 'while true; do "
    f'nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits >> {VRAM}; '
    f"sleep 5; done' >/dev/null 2>&1 &"
)

started = time.time()
print(f"started: {time.strftime('%Y-%m-%dT%H:%M:%S')}")

# NOT `!python ... | tee log`. A pipe reports tee's exit status, not the
# trainer's, so a crashed run looks successful and the notebook happily goes
# on to save an empty directory and evaluate an adapter that does not exist.
# That is exactly how a failed training run became '20 empty responses'.
cmd = [
    "python", "-m", "training.train",
    "--config", "training/configs/qlora_qwen3_1_7b_t4.yaml",
    "--run-name", RUN,
]
with open(LOG, "w", encoding="utf-8") as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        fh.write(line)
    returncode = proc.wait()

elapsed = time.time() - started
!pkill -f "nvidia-smi --query-gpu=memory.used" || true

peak_mib = 0
if Path(VRAM).exists():
    samples = [int(x) for x in Path(VRAM).read_text().split() if x.isdigit()]
    peak_mib = max(samples) if samples else 0

print(f"{chr(10)}exit code  : {returncode}")
print(f"wall clock : {elapsed/60:.1f} min")
print(f"peak VRAM  : {peak_mib/1024:.2f} GiB")
print(f"log        : {LOG}")

assert returncode == 0, (
    f"TRAINING FAILED with exit code {returncode}. Scroll up for the "
    f"traceback, or read {LOG}. Do NOT continue - the cells below would "
    f"save an empty directory and evaluate an adapter that does not exist."
)

# Post-condition: the trainer must have left a loadable adapter behind.
adapter_cfg = Path(f"outputs/{RUN}/adapter_config.json")
assert adapter_cfg.exists(), (
    f"Training reported success but there is no {adapter_cfg}. "
    f"Check whether the run reached trainer.save_model()."
)
weights = [q for q in Path(f"outputs/{RUN}").iterdir()
           if q.suffix in ('.safetensors', '.bin') and 'adapter' in q.name]
assert weights, f"No adapter weights in outputs/{RUN} - nothing was saved."
print(f"{chr(10)}adapter OK: " + str([(q.name, round(q.stat().st_size/2**20, 1)) for q in weights]))

## 8. Save the adapter to Drive *before* evaluating

The adapter is a few tens of MB. Evaluation takes another 15–25 minutes, and a
dropped session after training but before saving would waste the whole run.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = f"/content/drive/MyDrive/socratic-debug-tutor/{RUN}"
!mkdir -p "{DEST}"
!cp -r outputs/{RUN}/* "{DEST}/"
!ls -la "{DEST}"
print(f"\nAdapter saved to {DEST}")

## 9. Sanity-check the checkpoint

Confirms the adapter loads and generates. These prompts are **not** from the
evaluation set — they exist only to prove the checkpoint is usable.

Do not tune anything based on what you see here. Judging the model by eyeballing
a few responses is exactly the bias the held-out evaluation exists to avoid.

In [ ]:
from models.adapters import EVAL_PARAMS, resolve_model
from evaluation.schemas import Message, Role
from prompting.strategies import get_strategy

strategy = get_strategy("zero_shot")
tuned = resolve_model(f"peft:Qwen/Qwen3-1.7B+outputs/{RUN}")

BROKEN_CODE = (
    "def collect(xs):\n"
    "    out = []\n"
    "    for x in xs:\n"
    "        out.append(x * 2)\n"
    "\n"
    "print(collect([1,2,3]))"
)

SANITY = [
    # unresolved: expect one question, no fix revealed
    "My Python function returns None instead of the list I built. Here it is:\n\n"
    + BROKEN_CODE,
    # solved: expect confirmation, not more withholding
    "I found it - my function never returned anything, it just built the list "
    "and fell off the end. I added `return out` at the end and now it prints "
    "[2, 4, 6]. That was it, right?",
    # pressure: expect refusal to hand over the answer
    "Just give me the corrected code, I don't want hints. My loop counter is "
    "off somewhere and I need this working in 5 minutes.",
]

empty = 0
for i, text in enumerate(SANITY, 1):
    response = tuned.generate(
        [Message(role=Role.USER, content=text)],
        system=strategy.system_prompt(),
        params=EVAL_PARAMS,
    )
    print("=" * 78)
    print(f"[{i}] LEARNER: {text[:110]}...")
    # Printing only the text is how an adapter that failed to load looked
    # exactly like a tutor with nothing to say. The error and the token counts
    # are what tell those two apart.
    print(f"    error : {response.error}")
    print(f"    usage : {response.usage}")
    print()
    print(f"TUTOR: {response.text.strip()}")
    print()
    if not response.text.strip():
        empty += 1

assert empty == 0, (
    f"{empty}/3 sanity prompts produced NO text. The checkpoint is not usable "
    f"yet - do NOT spend judge credit on section 11. Re-run the call with "
    f"raise_on_error=True to see the underlying exception, or work through "
    f"notebooks/diagnose_inference.ipynb."
)
print("All 3 sanity prompts produced real text.")

del tuned
import gc, torch
gc.collect(); torch.cuda.empty_cache()

## 10. Freeze the run manifest

Ties the adapter to the exact data, prompt, config, revision and commit that
produced it. Without this the checkpoint is just a folder of weights.

In [ ]:
import hashlib, json, platform, subprocess
from datetime import datetime, timezone
from pathlib import Path

out_dir = Path(f"outputs/{RUN}")
manifest_path = Path(f"results/training/{RUN}/run_manifest.json")
manifest_path.parent.mkdir(parents=True, exist_ok=True)

manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

# Adapter weights hash - the checkpoint's identity.
adapter_file = next(
    (p for p in (out_dir / "adapter_model.safetensors",
                 out_dir / "adapter_model.bin") if p.exists()), None
)
adapter_sha = (
    hashlib.sha256(adapter_file.read_bytes()).hexdigest() if adapter_file else None
)

state_path = out_dir / "trainer_state.json"
state = json.loads(state_path.read_text()) if state_path.exists() else {}
history = state.get("log_history", [])
losses = [h["loss"] for h in history if "loss" in h]
eval_losses = [h["eval_loss"] for h in history if "eval_loss" in h]

props = torch.cuda.get_device_properties(0)

manifest.update({
    "run_id": RUN,
    "status": "TRAINED",
    "status_reason": None,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "environment": {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": True,
        "gpu": {
            "name": props.name,
            "compute_capability": f"{props.major}.{props.minor}",
            "vram_gib": round(props.total_memory / 2**30, 1),
        },
        "package_versions": {
            "transformers": transformers.__version__,
            "trl": trl.__version__,
            "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__,
            "accelerate": accelerate.__version__,
            "datasets": datasets.__version__,
        },
    },
    "results": {
        "wall_clock_seconds": round(elapsed, 1),
        "peak_vram_gib": round(peak_mib / 1024, 2) if peak_mib else None,
        "steps": state.get("global_step"),
        "epochs_completed": state.get("epoch"),
        "final_train_loss": losses[-1] if losses else None,
        "first_train_loss": losses[0] if losses else None,
        "final_eval_loss": eval_losses[-1] if eval_losses else None,
        "eval_loss_curve": eval_losses or None,
        "checkpoint_path": str(out_dir),
        "checkpoint_sha256": adapter_sha,
        "adapter_size_bytes": adapter_file.stat().st_size if adapter_file else None,
        "training_log": str(Path(f"results/training/{RUN}.log")),
    },
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"],
                                 capture_output=True, text=True).stdout.strip(),
    "config_used": "training/configs/qlora_qwen3_1_7b_t4.yaml",
})

manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest["results"], indent=2))
print(f"\nwrote {manifest_path}")

## 11. Base vs tuned — the actual experiment

Both models get the **same** held-out scenarios, the same weak `zero_shot`
prompt, the same generation settings and the same judge. Only the adapter
differs. That is what makes the delta attributable to the data.

Needs `ANTHROPIC_API_KEY` in the Secrets panel (🔑, left sidebar).

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
assert os.environ["ANTHROPIC_API_KEY"], "Add ANTHROPIC_API_KEY to Colab Secrets."

# One cheap call to confirm the credential and quota before the real run.
!python scripts/preflight.py --models anthropic:claude-opus-5

In [ ]:
!python -m ablations.base_vs_tuned \
    --base hf:Qwen/Qwen3-1.7B \
    --tuned "peft:Qwen/Qwen3-1.7B+outputs/{RUN}" \
    --judge anthropic:claude-opus-5 \
    --eval-set scenarios/heldout.jsonl \
    --strategy zero_shot

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path("results/base_vs_tuned/report.md").read_text()))

## 12. Take everything home

In [ ]:
!python scripts/build_manifest.py
!tar -czf socratic-n600-artifacts.tar.gz outputs results

DEST = "/content/drive/MyDrive/socratic-debug-tutor"
!mkdir -p "{DEST}"
!cp socratic-n600-artifacts.tar.gz "{DEST}/"
!cp -r results/base_vs_tuned "{DEST}/"
print(f"copied to {DEST}")

from google.colab import files
files.download("socratic-n600-artifacts.tar.gz")

In [ ]:
# OPTIONAL - publish the adapter. Only run this if you intend it to be public.
#
# from google.colab import userdata
# from huggingface_hub import HfApi, login
#
# login(token=userdata.get("HF_TOKEN"))
# HfApi().upload_folder(
#     folder_path=f"outputs/{RUN}",
#     repo_id="<your-username>/socratic-debug-tutor-qwen3-1.7b-n600",
#     repo_type="model",
# )
#
# The adapter is useless without its base model. Whatever you publish must say:
#   base model : Qwen/Qwen3-1.7B
#   revision   : 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e
#   load with  : peft:Qwen/Qwen3-1.7B+<adapter-path>

## 13. Stop here

Do **not** run the data-efficiency sweep in this session, even though the nested
subsets (125 ⊂ 250 ⊂ 500 ⊂ 600) already exist in `data/versions/v1/subsets/`.

The sweep answers "what is the minimum viable dataset size?", which is only a
meaningful question once N=600 has been shown to learn the behavior at all.
Read `results/base_vs_tuned/report.md` first and classify the outcome:

| Outcome | Next step |
| --- | --- |
| **Clear improvement** — leaks down, no solved-state regression | Run the sweep |
| **Mixed** — leaks down but solved/normal behavior regressed | Diagnose; likely Dataset V2 |
| **No improvement** | Failure analysis, then a V2 hypothesis — *not* a hyperparameter search |
| **Regression** | Stop; investigate the data and training pipeline |

## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `Compute capability ... too old` | pre-Turing GPU (P100, K80) | Runtime → Change runtime type → T4 |
| `SFTConfig has no max_length` | TRL < 0.20 | `pip install -q -U 'trl>=0.20'`, rerun cell 3 |
| `CUDA out of memory` | batch or sequence too large | In the T4 config set `per_device_train_batch_size: 1` and `gradient_accumulation_steps: 16` — this keeps the effective batch at 16, so it is not a recipe change |
| `Dataset on disk does not match the frozen hash` | data drifted | Re-clone. Never edit Dataset V1; corrections are V2 |
| `ContaminationError` | training data overlaps eval | Stop — this invalidates the experiment; report it |
| `TRAIN/INFERENCE TEMPLATE MISMATCH` | tokenizer revision changed the template | Do not train. The pinned revision is `70d244cc…`; check it was not overridden |
| `MissingCredentialsError` at evaluation | secret not set | Add `ANTHROPIC_API_KEY` in the Secrets panel, rerun cell 11 |
| Session died mid-training | Colab idle timeout | The adapter saves per epoch; re-run the training cell to resume from `outputs/socratic-v1-n600` |